# 04 — Inference Demo

Demo inference Donut trên ảnh biên lai + so sánh với PaddleOCR.

In [ ]:
import torch, json, os
from PIL import Image
import matplotlib.pyplot as plt
from transformers import DonutProcessor, VisionEncoderDecoderModel

CHECKPOINT = 'results/e2_donut/checkpoints/mcocr'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

try:
    processor = DonutProcessor.from_pretrained(CHECKPOINT)
    model = VisionEncoderDecoderModel.from_pretrained(CHECKPOINT).to(device)
    model.eval()
    print(f'Model loaded from {CHECKPOINT}')
except Exception as e:
    print(f'Chua co checkpoint: {e}')

In [ ]:
import sys, re
sys.path.insert(0, '..')
from scripts.utils import parse_donut_output, FIELDS

def predict(image_path):
    image = Image.open(image_path).convert('RGB')
    pixel_values = processor(image, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        gen = model.generate(
            pixel_values,
            max_length=model.config.decoder.max_position_embeddings,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )
    text = processor.tokenizer.decode(gen[0], skip_special_tokens=False)
    return parse_donut_output(text)

In [ ]:
# Inference tren 5 anh test
test_dir = 'data/mc-ocr/donut_format/test'
metadata_path = os.path.join(test_dir, 'metadata.jsonl')

if os.path.exists(metadata_path):
    with open(metadata_path, 'r', encoding='utf-8') as f:
        records = [json.loads(l) for l in f if l.strip()][:5]

    fig, axes = plt.subplots(1, len(records), figsize=(4*len(records), 6))
    if len(records) == 1: axes = [axes]
    for ax, rec in zip(axes, records):
        img_path = os.path.join(test_dir, rec['file_name'])
        if os.path.exists(img_path):
            ax.imshow(Image.open(img_path))
            pred = predict(img_path)
            info = '\n'.join(f'{k}: {v}' for k, v in pred.items() if v)
            ax.set_xlabel(info, fontsize=7)
        ax.set_title(rec['file_name'], fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()
else:
    print('Chua co test data')

In [ ]:
# So sanh Donut vs PaddleOCR tren cung anh
# from paddleocr import PaddleOCR
# from scripts.baseline_paddleocr import extract_fields_rule, run_paddleocr_on_image
# 
# ocr = PaddleOCR(use_angle_cls=True, lang='vi', show_log=False)
# img_path = 'data/mc-ocr/donut_format/test/<ten_anh>.jpg'
# 
# # PaddleOCR
# text_lines = run_paddleocr_on_image(ocr, img_path)
# paddle_pred = extract_fields_rule(text_lines)
# 
# # Donut
# donut_pred = predict(img_path)
# 
# print('PaddleOCR:', paddle_pred)
# print('Donut:    ', donut_pred)

## Nhan xet

- **Donut:** ...
- **So voi PaddleOCR:** ...
- **Truong hop kho:** ...